# Polyfun with pre-computed priors

In [3]:
from pyprojroot.here import here
import pandas as pd

Annotate SNPs with [pre-computed priors](https://github.com/omerwe/polyfun/wiki/1.-Computing-prior-causal-probabilities-with-PolyFun#polyfun-approach-1-using-precomputed-prior-causal-probabilities-based-on-a-meta-analysis-of-15-uk-biobank-traits). There are very few missing SNPs apart from chrX, so allow missing.

In [11]:
%%bash -s {here()}
here=$1
mkdir $here/data/processed/snpvar

python $here/vendor/polyfun/extract_snpvar.py \
    --sumstats $here/data/processed/sumstats/mdd2025_eur_hg19_z.tsv \
    --allow-missing \
    --out $here/data/processed/snpvar/mdd2025_eur_hg19_snpvar.gz    

[INFO]  Loading sumstats files...
[INFO]  Done in 25.86 seconds
[INFO]  Loading meta-analyzed per-SNP-h2 files...
[INFO]  Done in 63.06 seconds
[INFO]  Merging sumstats with per-SNP h2 data...
[INFO]  Flipping the Z-sign of 5658139 SNPs that A1 in sumstats = A2 in the per-SNP h2 data
[INFO]  Done in 20.49 seconds
[WARNING]  Not all SNPs in the SNPs file were found in the meta file. Wrote a list of missing SNPs to /home/madams23/Projects/cvd-mh-loci/data/processed/snpvar/mdd2025_eur_hg19_snpvar.gz.miss.gz
[INFO]  Writing output file to /home/madams23/Projects/cvd-mh-loci/data/processed/snpvar/mdd2025_eur_hg19_snpvar.gz


mkdir: cannot create directory ‘/home/madams23/Projects/cvd-mh-loci/data/processed/snpvar’: File exists


In [2]:
%%bash -s {here()}
here=$1

mkdir -p $here/data/cache/ld
mkdir -p $here/data/results/PolyFun

python $here/vendor/polyfun/finemapper.py \
    --geno $here/data/processed/reference/all_h19_EUR_chr11_61000000-63000000 \
    --sumstats $here/data/processed/snpvar/mdd2025_eur_hg19_snpvar.gz \
    --n 1577200 \
    --chr 11 \
    --start 61400000 \
    --end 61700000 \
    --method susie \
    --max-num-causal 5 \
    --cache-dir $here/data/cache/ld \
    --out $here/data/results/PolyFun/mdd2025_eur_polyfun_susie_chr11_61000000-63000000.gz \
    --memory 24 \
    --threads 6

*********************************************************************
* Fine-mapping Wrapper
* Version 1.0.0
* (C) 2019-2024 Omer Weissbrod
*********************************************************************

[INFO]  Loading sumstats file...
[INFO]  Loaded sumstats for 351143 SNPs in 12.13 seconds
[INFO]  cffi mode is CFFI_MODE.ANY
[DEBUG]  Looking for R home with: R RHOME
[INFO]  R home found: /home/madams23/Projects/cvd-mh-loci/.pixi/envs/polyfun/lib/R
[DEBUG]  Looking for LD_LIBRARY_PATH with: /home/madams23/Projects/cvd-mh-loci/.pixi/envs/polyfun/lib/R/bin/Rscript -e cat(Sys.getenv("LD_LIBRARY_PATH"))
[INFO]  R library path: 
[INFO]  LD_LIBRARY_PATH: 
[DEBUG]  cffi mode is InterfaceType.API
[INFO]  Default options to initialize R: rpy2, --quiet, --no-save
[INFO]  R is already initialized. No need to initialize.
[INFO]  Computing LD from plink fileset /home/madams23/Projects/cvd-mh-loci/data/processed/reference/all_h19_EUR_chr11_61000000-63000000 chromosome 11 region 61400000-6170

Mapping files: 100%|██████████| 3/3 [00:00<00:00, 11.58it/s]


[INFO]  Found 8385 SNPs in target region. Computing LD in 1 chunks...


100%|██████████| 1/1 [00:01<00:00,  1.80s/it]


[INFO]  Done in 2.54 seconds
[INFO]  Saving LD file /home/madams23/Projects/cvd-mh-loci/data/cache/ld/all_h19_EUR_chr11_61000000-63000000.11.61400000.61700000.npz
[INFO]  Done in 1.66 seconds
[INFO]  Starting functionally-informed SuSiE fine-mapping for chromosome 11 BP 61400000-61700000 (589 SNPs)
[INFO]  Using susieR::susie_suff_stat()
[INFO]  Done in 0.28 seconds
[INFO]  Using rpy2 version 3.5.11
[INFO]  Writing fine-mapping results to /home/madams23/Projects/cvd-mh-loci/data/results/PolyFun/mdd2025_eur_polyfun_susie_chr11_61000000-63000000.gz


In [6]:
susie = fads = pd.read_csv(
    here("data/results/PolyFun/mdd2025_eur_polyfun_susie_chr11_61000000-63000000.gz"),
    sep = "\t",
    compression = "infer"
)

cr = (
    susie
    .loc[susie["CREDIBLE_SET"].ne(0), ["CREDIBLE_SET", "SNP", "P", "PIP"]]
    .sort_values(by=["CREDIBLE_SET", "PIP"], ascending=[True, False])
)

cr

,CREDIBLE_SET,SNP,P,PIP
1,1,rs174561,7.925300e-20,0.290706
2,1,rs174544,1.702760e-20,0.232573
5,1,rs99780,4.950110e-20,0.117866
7,1,rs174583,3.261250e-20,0.099338
8,1,rs174548,3.738480e-21,0.060067
11,1,rs174549,5.420130e-21,0.041348
12,1,rs28456,5.420130e-21,0.035037
13,1,rs174574,4.950110e-20,0.029461
15,1,rs174560,1.178850e-20,0.024314
18,1,rs174535,9.435750e-19,0.009871
